In [17]:
import psycopg as pg
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

True

In [79]:
# leggiamo user e pass da .env

host = "raspberry-mf.local"
dbname = "seriea_reference"
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")

conn = pg.connect(host=host, dbname=dbname, user=user, password=password)
conn.autocommit = True
cursor = conn.cursor()

In [3]:
all_games = pd.read_csv('stats_reference/static/all_games.csv')
games_details = pd.read_pickle('stats_reference/static/games_details.pkl')

print(all_games.columns, games_details.columns)

Index(['Team ID', 'Anno', 'Giornata', 'Data', 'DataOra', 'Squadra Casa',
       'Punteggio', 'Squadra Trasferta', 'Link Boxscore'],
      dtype='str') Index(['Partita', 'Squadra', 'Titolare', 'Numero', 'Giocatore', 'Punti',
       'Minuti', 'Falli Commessi', 'Falli Subiti', 'Tiri da 2 Realizzati',
       'Tiri da 2 Tentati', '% da 2', 'Schiacciate', 'Tiri da 3 Realizzati',
       'Tiri da 3 Tentati', '% da 3', 'Tiri Liberi Realizzati',
       'Tiri da Liberi Tentati', '% Liberi', 'Rimbalzi Dif', 'Rimbalzi Off',
       'Rimbalzi Tot', 'Stoppate Date', 'Stoppate Subite', 'Palle Perse',
       'Palle Recuperate', 'Assist', 'Valutazione Lega', 'Valutazione OER',
       '+/-'],
      dtype='object')


In [ ]:
all_games_cols = {
    'Team ID': 'team_id',
    'Anno': 'season',
    'Giornata': 'matchday',
    'DataOra': 'game_date',
    'Squadra Casa': 'home_team',
    'Punteggio': 'score',
    'Squadra Trasferta': 'away_team',
    'Link Boxscore': 'boxscore_link',
}

games_details_cols = {
    'Partita': 'game',
    'Squadra': 'team',
    'Titolare': 'starter',
    'Numero': 'number',
    'Giocatore': 'player',
    'Punti': 'points',
    'Minuti': 'minutes',
    'Falli Commessi': 'fouls_committed',
    'Falli Subiti': 'fouls_received',
    'Tiri da 2 Realizzati': 'two_pts_made',
    'Tiri da 2 Tentati': 'two_pts_attempted',
    '% da 2': 'two_pts_perc',
    'Schiacciate': 'dunks',
    'Tiri da 3 Realizzati': 'three_pts_made',
    'Tiri da 3 Tentati': 'three_pts_attempted',
    '% da 3': 'three_pts_perc',
    'Tiri Liberi Realizzati': 'ft_made',
    'Tiri da Liberi Tentati': 'ft_attempted',
    '% Liberi': 'ft_perc',
    'Rimbalzi Dif': 'def_rebounds',
    'Rimbalzi Off': 'off_rebounds',
    'Rimbalzi Tot': 'tot_rebounds',
    'Stoppate Date': 'blocks_given',
    'Stoppate Subite': 'blocks_received',
    'Palle Perse': 'turnovers',
    'Palle Recuperate': 'steals',
    'Assist': 'assists',
    'Valutazione Lega': 'league_rating',
    'Valutazione OER': 'oer_rating',
    '+/-': 'plus_minus',
}

all_games = all_games.rename(columns=all_games_cols)
# Scarta la colonna 'Data' (non presente in tabella)
all_games = all_games.drop(columns=['Data'], errors='ignore')

games_details = games_details.rename(columns=games_details_cols)

print(all_games.columns.tolist())
print(games_details.columns.tolist())


['team_id', 'season', 'matchday', 'datetime', 'home_team', 'score', 'away_team', 'boxscore_link']
['game', 'team', 'starter', 'number', 'player', 'points', 'minutes', 'fouls_committed', 'fouls_received', 'two_pts_made', 'two_pts_attempted', 'two_pts_perc', 'dunks', 'three_pts_made', 'three_pts_attempted', 'three_pts_perc', 'ft_made', 'ft_attempted', 'ft_perc', 'def_rebounds', 'off_rebounds', 'tot_rebounds', 'blocks_given', 'blocks_received', 'turnovers', 'steals', 'assists', 'league_rating', 'oer_rating', 'plus_minus']


In [ ]:
all_games.head(2)

In [ ]:
games_details.head(2)

In [ ]:
CREATE_ALL_GAMES = """
CREATE TABLE IF NOT EXISTS games (
    id          SERIAL PRIMARY KEY,
    team_id     INTEGER,
    season      VARCHAR(9),
    matchday    VARCHAR(100),
    game_date    TIMESTAMPTZ,
    home_team   VARCHAR(100),
    score       VARCHAR(100),
    away_team   VARCHAR(100),
    boxscore_link TEXT
);
"""

CREATE_GAMES_DETAILS = """
CREATE TABLE IF NOT EXISTS games_details (
    id                  SERIAL PRIMARY KEY,
    game_id             BIGINT REFERENCES games(id) ON DELETE CASCADE,
    team                VARCHAR(100),
    starter             BOOLEAN,
    number              INTEGER,
    player              VARCHAR(100),
    points              NUMERIC(14,6),
    minutes             NUMERIC(14,6),
    fouls_committed     INTEGER,
    fouls_received      INTEGER,
    two_pts_made        INTEGER,
    two_pts_attempted   INTEGER,
    two_pts_perc        NUMERIC(14,6),
    dunks               INTEGER,
    three_pts_made      INTEGER,
    three_pts_attempted INTEGER,
    three_pts_perc      NUMERIC(14,6),
    ft_made             INTEGER,
    ft_attempted        INTEGER,
    ft_perc             NUMERIC(14,6),
    def_rebounds        INTEGER,
    off_rebounds        INTEGER,
    tot_rebounds        INTEGER,
    blocks_given        INTEGER,
    blocks_received     INTEGER,
    turnovers           INTEGER,
    steals              INTEGER,
    assists             INTEGER,
    league_rating       INTEGER,
    oer_rating          NUMERIC(14,6),
    plus_minus          INTEGER,
    created_at			TIMESTAMPTZ DEFAULT NOW()
);
"""

In [ ]:
# --- Pulizia datetime ---
# Formato atteso: "13/10/1974 00:00", valori non validi (es. "00/00/0000 00:00") -> 1900-01-01
games_df = all_games.copy()
games_df['game_date'] = pd.to_datetime(
    games_df['game_date'], format='%d/%m/%Y %H:%M', errors='coerce'
).fillna(pd.Timestamp('1900-01-01'))

# --- Insert nella tabella games con RETURNING id ---
game_id_map = {}  # boxscore_link -> id

print(all_games.shape, games_details.shape)

(20138, 8) (352271, 30)


In [ ]:
INSERT_GAME = """
    INSERT INTO games (team_id, season, matchday, game_date, home_team, score, away_team, boxscore_link)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING id
"""

params = [
    (
        row['team_id'],
        row['season'],
        row['matchday'],
        row['game_date'].to_pydatetime(),
        row['home_team'],
        row['score'],
        row['away_team'],
        row['boxscore_link'],
    )
    for _, row in games_df.iterrows()
]

game_id_map = {}  # boxscore_link -> id

try:
    cursor.executemany(INSERT_GAME, params, returning=True)
    for link in games_df['boxscore_link']:
        game_id = cursor.fetchone()[0]
        game_id_map[link] = game_id
        cursor.nextset()
    print(f"Inserite {len(game_id_map)} righe in games.")
except Exception as e:
    conn.rollback()
    raise e

Inserite 20138 righe in games.


In [47]:
query = """select * from public.games"""
df_real_games = pd.read_sql(query, conn)
df_real_games = df_real_games[['id', 'legabasket_game_id']]
df_real_games.rename(columns={'id': 'game_id'}, inplace=True)

/var/folders/yc/vtlqf6s13yn4ckx3m6tzp_gr0000gn/T/ipykernel_51735/2764941970.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_real_games = pd.read_sql(query, conn)


In [49]:
games_details.head(2)

,game,team,starter,number,player,points,minutes,fouls_committed,fouls_received,two_pts_made,...,off_rebounds,tot_rebounds,blocks_given,blocks_received,turnovers,steals,assists,league_rating,oer_rating,plus_minus
0,/game/52194/basket_brescia-dietor_bologna_81-92,Basket Brescia,*,0,Gilardi Enrico,11,37,3,2,3,...,5,5,0,0,4,3,6,11,0.63,0
1,/game/52194/basket_brescia-dietor_bologna_81-92,Basket Brescia,*,0,Sitton Charlie,24,38,2,5,10,...,5,9,0,0,2,5,0,30,1.07,0


In [53]:
games_details.loc[0, 'game'].str.split('/')

0    [, game, 52194, basket_brescia-dietor_bologna_...
0    [, game, 1672952, banco_di_sardegna_sassari-do...
0    [, game, 1672236, alma_trieste-segafredo_virtu...
0    [, game, 23362, gevi-napoli-basket-a-x-armani-...
0    [, game, 23698, virtus-segafredo-bologna-a-x-a...
0    [, game, 23703, nutribullet-treviso-basket-una...
0    [, game, 23746, givova-scafati-openjobmetis-va...
0    [, game, 23780, banco-di-sardegna-sassari-gevi...
0    [, game, 23823, carpegna-prosciutto-pesaro-tez...
0    [, game, 23703, nutribullet-treviso-basket-una...
0    [, game, 24086, dolomiti-energia-trentino-vano...
Name: game, dtype: object

In [71]:
games_details['legabasket_game_id'] = games_details['game'].apply(lambda x: int(x.split('/')[2]))

insert_games_details = games_details.copy(deep=True)
print(insert_games_details.shape[0])
insert_games_details = insert_games_details.merge(df_real_games, on='legabasket_game_id', how='inner')
print(insert_games_details.shape[0])

insert_games_details['map_game_id'] = insert_games_details['game'].map(game_id_map)
insert_games_details = insert_games_details.dropna(subset=['map_game_id'])

insert_games_details.drop(columns=['legabasket_game_id', 'map_game_id'], inplace=True)
print(insert_games_details.shape[0])

352271
352271
346413


In [72]:
insert_games_details.groupby('game_id').count().sort_values('team', ascending=False).head(10)

,game,team,starter,number,player,points,minutes,fouls_committed,fouls_received,two_pts_made,...,off_rebounds,tot_rebounds,blocks_given,blocks_received,turnovers,steals,assists,league_rating,oer_rating,plus_minus
game_id,,,,,,,,,,,,,,,,,,,,,
20001,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16555,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16547,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16544,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16541,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16540,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16538,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16537,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28
16532,28,28,28,28,28,28,28,28,28,28,...,28,28,28,28,28,28,28,28,28,28


In [74]:
df_real_games[~df_real_games['game_id'].isin(insert_games_details['game_id'].unique())]

,game_id,legabasket_game_id
0,1,7220
1,2,3766
2,3,3767
3,4,3770
4,5,3769
...,...,...
20133,20134,24448
20134,20135,24445
20135,20136,24449
20136,20137,24446


In [75]:
insert_games_details[insert_games_details['game_id'] == 7220]

,game,team,starter,number,player,points,minutes,fouls_committed,fouls_received,two_pts_made,...,tot_rebounds,blocks_given,blocks_received,turnovers,steals,assists,league_rating,oer_rating,plus_minus,game_id
40121,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,*,0,Benatti Maurizio,2,23,1,0,1,...,3,0,0,2,0,0,0,0.40,0,7220
40122,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,*,0,Wright Brad,17,35,5,4,6,...,4,0,0,3,5,1,19,1.10,0,7220
40123,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,,0,Ruggeri Massimo,0,0,0,0,0,...,0,0,0,0,0,0,0,0.00,0,7220
40124,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,,0,Myers Carlton,0,0,0,0,0,...,0,0,0,0,0,0,0,0.00,0,7220
40125,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,,0,Neri Emiliano,2,17,3,0,1,...,1,0,0,3,0,0,-3,0.50,0,7220
40126,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,,0,Carboni Alfredo,17,20,2,4,3,...,4,0,0,1,1,0,20,1.55,0,7220
40127,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,*,0,Tufano Giacomantonio,0,10,2,0,0,...,4,0,1,0,0,0,-3,0.00,0,7220
40128,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,*,0,Ambrassa Fabrizio,15,37,2,4,3,...,6,0,2,1,1,0,17,1.36,0,7220
40129,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,,0,Ferro Maurizio,15,19,4,5,2,...,0,0,0,1,0,0,9,1.25,0,7220
40130,/game/53575/marr_rimini-b__popolare_sassari_81-73,Marr Rimini,*,0,Smith Mark,13,39,2,7,3,...,6,0,0,3,3,1,19,0.87,0,7220


In [76]:
insert_games_details['starter'].unique()

array(['*', ''], dtype=object)

In [77]:
# starter: '*' -> True, qualsiasi altra cosa -> False
insert_games_details['starter'] = insert_games_details['starter'] == '*'

# number: stringhe vuote e '-' -> pd.NA, poi nullable integer (NULL in postgres)
insert_games_details['number'] = pd.to_numeric(
    insert_games_details['number'].replace({'': None, '-': None}),
    errors='coerce'
).astype(pd.Int64Dtype())

insert_games_details[['starter', 'number']].head(10)

,starter,number
0,True,0
1,True,0
2,False,0
3,False,0
4,False,0
5,False,0
6,True,0
7,False,0
8,True,0
9,True,0


In [82]:
INSERT_GAME_DETAIL = """
    INSERT INTO games_details (
        game_id, team, starter, number, player, points, minutes,
        fouls_committed, fouls_received,
        two_pts_made, two_pts_attempted, two_pts_perc,
        dunks,
        three_pts_made, three_pts_attempted, three_pts_perc,
        ft_made, ft_attempted, ft_perc,
        def_rebounds, off_rebounds, tot_rebounds,
        blocks_given, blocks_received,
        turnovers, steals, assists,
        league_rating, oer_rating, plus_minus
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cols = [
    'game_id', 'team', 'starter', 'number', 'player', 'points', 'minutes',
    'fouls_committed', 'fouls_received',
    'two_pts_made', 'two_pts_attempted', 'two_pts_perc',
    'dunks',
    'three_pts_made', 'three_pts_attempted', 'three_pts_perc',
    'ft_made', 'ft_attempted', 'ft_perc',
    'def_rebounds', 'off_rebounds', 'tot_rebounds',
    'blocks_given', 'blocks_received',
    'turnovers', 'steals', 'assists',
    'league_rating', 'oer_rating', 'plus_minus'
]

_NULL_STRINGS = {'', '-', 'n/a', 'na', 'null', 'none'}

def to_none(v):
    """Converte pd.NA, np.nan e stringhe non numeriche in None (NULL postgres)."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(v, str) and v.strip().lower() in _NULL_STRINGS:
        return None
    return v

params = [
    tuple(to_none(v) for v in row)
    for row in insert_games_details[cols].itertuples(index=False, name=None)
]

try:
    cursor.executemany(INSERT_GAME_DETAIL, params)
    print(f"Inserite {len(params)} righe in games_details.")
except Exception as e:
    conn.rollback()
    raise e


Inserite 346413 righe in games_details.
